# <span style="color:green"> INDIVIDUAL EXPLORATORY ANALYSIS</span>

## <span style="color:green"> PACKAGES USED </span> ##

In [5]:
import pandas as pd
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io
import base64
from pathlib import Path
import gc
from matplotlib.ticker import FuncFormatter

## <span style="color:green"> DATASET IMPORT AND DIRECTION OF ADA STUDIES </span> ##

In [2]:
dataset_final = pd.read_parquet(
    "../../data/final_dataset/dataset_final.parquet"
)

print(dataset_final.shape)

for i, column in enumerate(
    dataset_final.columns,
    start=1
):
    print(f"{i}. {column}")
    

(1852394, 22)
1. NID_ALPHA
2. TRANS_NUM_CARD_FEWF
3. RECEIVE_LOC_FEWF
4. RECEIVE_CATEGORY_OHEWI
5. TRANS_VALUE
6. SEND_GENDER_BE
7. SEND_LAT_REGISTER
8. SEND_LONG_REGISTER
9. SEND_POP_REGISTER
10. SEND_JOB_FEWF
11. RECEIVE_LAT
12. RECEIVE_LONG
13. TRANS_DAY
14. TRANS_WEEK_OHEWI
15. TRANS_YEAR_BE
16. TRANS_MONTH_SIN
17. TRANS_MONTH_COS
18. TRANS_HOUR_SIN
19. TRANS_HOUR_COS
20. SEND_NAME_FEWF
21. SEND_AGE
22. TARGET_OMEGA


## <span style="color:green"> BINARY ENCODING </span> ##

### <span style="color:white"> TARGET_OMEGA </span> ###

In [ ]:
# ============================================================
# 01. ANALYSIS SETTINGS
#
# FOR THE NEXT FEATURES, CHANGE ONLY THESE TWO LINES
# ============================================================

ENCODING_TYPE = "binary_encoding"
FEATURE_NAME = "target_omega"


# Actual column name inside the dataset
FEATURE_COLUMN = FEATURE_NAME.upper()


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / ENCODING_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


LOG_CHART_NAME = (
    f"{FEATURE_NAME}_class_distribution_log"
)


LOG_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{LOG_CHART_NAME}.png"
)


CUMULATIVE_CHART_NAME = (
    f"{FEATURE_NAME}_cumulative_fraud"
)


CUMULATIVE_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{CUMULATIVE_CHART_NAME}.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)

log_chart_exists = (
    LOG_CHART_PATH.exists()
)

cumulative_chart_exists = (
    CUMULATIVE_CHART_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and log_chart_exists
    and cumulative_chart_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )

    print(
        "No analysis or file creation is required."
    )

    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )

    print(
        "\nExisting files:"
    )

    print(
        HTML_PATH
    )

    print(
        LOG_CHART_PATH
    )

    print(
        CUMULATIVE_CHART_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )

    print(
        "=" * 100
    )

    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )

    print(
        "Logarithmic distribution chart:",
        "Already exists"
        if log_chart_exists
        else "Will be created"
    )

    print(
        "Cumulative fraud chart:",
        "Already exists"
        if cumulative_chart_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. VALIDATE THE FEATURE
    # ========================================================

    if feature.isna().any():

        raise ValueError(
            f"{FEATURE_COLUMN} contains missing values."
        )


    found_values = set(
        feature.unique()
    )


    if not found_values.issubset(
        {0, 1}
    ):

        raise ValueError(
            f"{FEATURE_COLUMN} contains values "
            f"different from 0 and 1: "
            f"{found_values}"
        )


    # ========================================================
    # 13. ABSOLUTE COUNT OF EACH CLASS
    # ========================================================

    class_count = (
        feature
        .value_counts()
        .reindex(
            [
                0,
                1
            ],
            fill_value=0
        )
    )


    non_fraud = int(
        class_count.loc[0]
    )


    fraud = int(
        class_count.loc[1]
    )


    total = int(
        class_count.sum()
    )


    # ========================================================
    # 14. PERCENTAGE PROPORTION OF EACH CLASS
    # ========================================================

    non_fraud_percentage = (
        non_fraud
        / total
        * 100
    )


    fraud_percentage = (
        fraud
        / total
        * 100
    )


    # ========================================================
    # 15. NON-FRAUD TO FRAUD RATIO
    # ========================================================

    if fraud > 0:

        non_fraud_fraud_ratio = (
            non_fraud
            / fraud
        )

    else:

        non_fraud_fraud_ratio = np.inf


    # ========================================================
    # 16. IMBALANCE RATIO (IR)
    #
    # Majority class / Minority class
    # ========================================================

    majority_class = int(
        class_count.max()
    )


    minority_class = int(
        class_count.min()
    )


    if minority_class > 0:

        imbalance_ratio = (
            majority_class
            / minority_class
        )

    else:

        imbalance_ratio = np.inf


    # ========================================================
    # 17. SHANNON ENTROPY
    # ========================================================

    proportions = (
        class_count
        / total
    )


    shannon_entropy = -sum(
        proportion
        * np.log2(
            proportion
        )

        for proportion in proportions

        if proportion > 0
    )


    # ========================================================
    # 18. FUNCTION TO CONVERT AN EXISTING PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 19. CREATE THE LOGARITHMIC DISTRIBUTION CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not log_chart_exists:

        labels = [
            "Non-fraud (0)",
            "Fraud (1)"
        ]


        values = [
            non_fraud,
            fraud
        ]


        fig, ax = plt.subplots(
            figsize=(
                8,
                6
            )
        )


        bars = ax.bar(
            labels,
            values
        )


        ax.set_yscale(
            "log"
        )


        ax.set_title(
            f"{FEATURE_COLUMN} class distribution "
            f"on a logarithmic scale"
        )


        ax.set_xlabel(
            "Class"
        )


        ax.set_ylabel(
            "Number of transactions"
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        # ----------------------------------------------------
        # DISPLAY VALUES ABOVE THE BARS
        # WITHOUT THOUSANDS SEPARATORS
        # ----------------------------------------------------

        for bar, value in zip(
            bars,
            values
        ):

            ax.text(
                bar.get_x()
                + bar.get_width() / 2,

                value,

                str(
                    value
                ),

                ha="center",
                va="bottom"
            )


        fig.tight_layout()


        fig.savefig(
            LOG_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nLogarithmic distribution chart created:"
        )

        print(
            LOG_CHART_PATH
        )


    else:

        print(
            "\nLogarithmic distribution chart already exists."
        )

        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 21. IDENTIFY FRAUDULENT TRANSACTION POSITIONS
    # ========================================================

    feature_array = (
        feature.to_numpy()
    )


    fraud_positions = (
        np.flatnonzero(
            feature_array == 1
        )
        + 1
    )


    # ========================================================
    # 21. BUILD THE CUMULATIVE FRAUD CURVE
    # ========================================================

    if len(
        fraud_positions
    ) > 0:

        cumulative_x = np.concatenate(
            (
                [
                    1
                ],

                fraud_positions,

                [
                    total
                ]
            )
        )


        cumulative_y = np.concatenate(
            (
                [
                    0
                ],

                np.arange(
                    1,
                    len(
                        fraud_positions
                    )
                    + 1
                ),

                [
                    len(
                        fraud_positions
                    )
                ]
            )
        )


    else:

        cumulative_x = np.array(
            [
                1,
                total
            ]
        )


        cumulative_y = np.array(
            [
                0,
                0
            ]
        )


    # ========================================================
    # 22. CREATE THE CUMULATIVE FRAUD CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not cumulative_chart_exists:

        fig, ax = plt.subplots(
            figsize=(
                12,
                6
            )
        )


        ax.step(
            cumulative_x,
            cumulative_y,
            where="post"
        )


        ax.set_title(
            "Cumulative fraud by transaction order"
        )


        ax.set_xlabel(
            "Transaction order"
        )


        ax.set_ylabel(
            "Cumulative number of fraudulent transactions"
        )


        ax.set_xlim(
            1,
            total
        )


        ax.set_ylim(
            bottom=0
        )


        ax.grid(
            alpha=0.3
        )


        # ----------------------------------------------------
        # AXIS FORMAT
        #
        # Without thousands separators.
        #
        # Example:
        # 500000
        #
        # Not:
        # 500,000
        # ----------------------------------------------------

        ax.xaxis.set_major_formatter(
            FuncFormatter(
                lambda x, pos:
                str(
                    int(
                        x
                    )
                )
            )
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(
                        y
                    )
                )
            )
        )


        fig.tight_layout()


        fig.savefig(
            CUMULATIVE_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nCumulative fraud chart created:"
        )

        print(
            CUMULATIVE_CHART_PATH
        )


    else:

        print(
            "\nCumulative fraud chart already exists."
        )

        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 23. CREATE THE HTML REPORT
    #
    # Only if the HTML does not already exist.
    # ========================================================

    if not html_exists:

        # ----------------------------------------------------
        # CONVERT THE PNG FILES TO BASE64
        #
        # At this point the PNG files either already existed
        # or were created during this execution.
        # ----------------------------------------------------

        log_chart_base64 = (
            image_to_base64(
                LOG_CHART_PATH
            )
        )


        cumulative_chart_base64 = (
            image_to_base64(
                CUMULATIVE_CHART_PATH
            )
        )


        # ====================================================
        # HTML CONTENT
        # ====================================================

        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1100px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 25px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 10px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 35px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.summary {{
    margin-bottom: 40px;
}}

</style>

</head>


<body>


<!-- ========================================================
     TITLE
========================================================= -->


<h1>
Individual Exploratory Analysis — {FEATURE_COLUMN}
</h1>


<p>

The variable <strong>{FEATURE_COLUMN}</strong>
corresponds to the binary target variable
of the dataset.

</p>


<ul>

<li>
<strong>0:</strong>
non-fraudulent transaction
</li>

<li>
<strong>1:</strong>
fraudulent transaction
</li>

</ul>


<!-- ========================================================
     1. ABSOLUTE CLASS COUNT
========================================================= -->


<h2>
1. Absolute count of each class
</h2>


<table>

<thead>

<tr>

<th>Class</th>

<th>Meaning</th>

<th>Count</th>

</tr>

</thead>


<tbody>


<tr>

<td>0</td>

<td>Non-fraud</td>

<td>{non_fraud}</td>

</tr>


<tr>

<td>1</td>

<td>Fraud</td>

<td>{fraud}</td>

</tr>


<tr>

<td>
<strong>Total</strong>
</td>

<td>-</td>

<td>
<strong>{total}</strong>
</td>

</tr>


</tbody>

</table>


<!-- ========================================================
     2. PERCENTAGE PROPORTION
========================================================= -->


<h2>
2. Percentage proportion of each class
</h2>


<table>

<thead>

<tr>

<th>Class</th>

<th>Meaning</th>

<th>Percentage</th>

</tr>

</thead>


<tbody>


<tr>

<td>0</td>

<td>Non-fraud</td>

<td>
{non_fraud_percentage:.6f}%
</td>

</tr>


<tr>

<td>1</td>

<td>Fraud</td>

<td>
{fraud_percentage:.6f}%
</td>

</tr>


</tbody>

</table>


<!-- ========================================================
     3. NON-FRAUD TO FRAUD RATIO
========================================================= -->


<h2>
3. Non-fraud to fraud ratio
</h2>


<p class="result">

{non_fraud_fraud_ratio:.2f}:1

</p>


<p>

There is approximately

<strong>

1 fraudulent transaction for every
{non_fraud_fraud_ratio:.2f}
non-fraudulent transactions.

</strong>

</p>


<!-- ========================================================
     4. IMBALANCE RATIO
========================================================= -->


<h2>
4. Imbalance Ratio (IR)
</h2>


<p class="result">

IR = {imbalance_ratio:.4f}

</p>


<p>

The Imbalance Ratio represents the ratio
between the number of observations
in the majority class and the number
of observations in the minority class.

</p>


<p>

The larger the IR value,
the greater the imbalance
between the classes.

</p>


<!-- ========================================================
     5. SHANNON ENTROPY
========================================================= -->


<h2>
5. Shannon Entropy
</h2>


<p class="result">

Entropy = {shannon_entropy:.6f} bits

</p>


<p>

For a binary variable,
Shannon Entropy ranges
from 0 to 1 bit.

</p>


<ul>

<li>

Values close to
<strong>0</strong>
indicate a greater concentration
of observations in a single class.

</li>


<li>

Values close to
<strong>1</strong>
indicate a greater balance
between the two classes.

</li>

</ul>


<!-- ========================================================
     6. LOGARITHMIC CLASS DISTRIBUTION
========================================================= -->


<h2>
6. Class distribution on a logarithmic scale
</h2>


<p>

The logarithmic scale facilitates
the visual comparison between the two classes
when there is a large difference
between their absolute frequencies.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{log_chart_base64}"
    alt="{FEATURE_COLUMN} class distribution on a logarithmic scale"
>

</div>


<!-- ========================================================
     7. CUMULATIVE FRAUD
========================================================= -->


<h2>
7. Cumulative fraud by transaction order
</h2>


<p>

The X axis represents the sequential position
of each transaction in the dataset.

Position 1 corresponds to the first transaction,
position 2 to the second transaction,
position 3 to the third transaction,
and so forth.

</p>


<p>

The Y axis represents the cumulative number
of fraudulent transactions.

Whenever an observation has
<strong>{FEATURE_COLUMN} = 1</strong>,
the cumulative value increases by one unit.

When an observation has
<strong>{FEATURE_COLUMN} = 0</strong>,
the cumulative value remains unchanged.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{cumulative_chart_base64}"
    alt="Cumulative fraud by transaction order"
>

</div>


<!-- ========================================================
     8. SUMMARY
========================================================= -->


<h2>
8. Summary of results
</h2>


<div class="summary">

<ul>


<li>

<strong>
Total observations:
</strong>

{total}

</li>


<li>

<strong>
Non-fraudulent transactions:
</strong>

{non_fraud}

</li>


<li>

<strong>
Fraudulent transactions:
</strong>

{fraud}

</li>


<li>

<strong>
Non-fraud percentage:
</strong>

{non_fraud_percentage:.6f}%

</li>


<li>

<strong>
Fraud percentage:
</strong>

{fraud_percentage:.6f}%

</li>


<li>

<strong>
Non-fraud to fraud ratio:
</strong>

{non_fraud_fraud_ratio:.2f}:1

</li>


<li>

<strong>
Imbalance Ratio:
</strong>

{imbalance_ratio:.4f}

</li>


<li>

<strong>
Shannon Entropy:
</strong>

{shannon_entropy:.6f} bits

</li>


</ul>

</div>


</body>

</html>
"""


        # ====================================================
        # 24. SAVE THE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )

        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )

        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 26. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature
    del feature_array
    del fraud_positions
    del cumulative_x
    del cumulative_y

    gc.collect()


    # ========================================================
    # 26. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "ANALYSIS COMPLETED"
    )

    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )

    print(
        HTML_PATH
    )


    print(
        "\nPNG files:"
    )

    print(
        LOG_CHART_PATH
    )

    print(
        CUMULATIVE_CHART_PATH
    )

All analysis files already exist.
No analysis or file creation is required.

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/target_omega

Existing files:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/target_omega/analysis_target_omega.html
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/target_omega/target_omega_class_distribution_log.png
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/target_omega/target_omega_cumulative_fraud.png


### <span style="color:black"> TRANS_YEAR_BE </span> ###

In [7]:
# ============================================================
# 01. ANALYSIS SETTINGS
#
# FOR THE NEXT FEATURES, CHANGE MAINLY THESE TWO LINES
# ============================================================

ENCODING_TYPE = "binary_encoding"
FEATURE_NAME = "trans_year_be"


# Actual column name inside the dataset
FEATURE_COLUMN = FEATURE_NAME.upper()


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / ENCODING_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


DISTRIBUTION_CHART_NAME = (
    f"{FEATURE_NAME}_distribution"
)


DISTRIBUTION_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{DISTRIBUTION_CHART_NAME}.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


distribution_chart_exists = (
    DISTRIBUTION_CHART_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and distribution_chart_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )

    print(
        "No analysis or file creation is required."
    )

    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )

    print(
        "\nExisting files:"
    )

    print(
        HTML_PATH
    )

    print(
        DISTRIBUTION_CHART_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )

    print(
        "=" * 100
    )

    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )

    print(
        "Distribution chart:",
        "Already exists"
        if distribution_chart_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. VALIDATE THE FEATURE
    # ========================================================

    if feature.isna().any():

        raise ValueError(
            f"{FEATURE_COLUMN} contains missing values."
        )


    found_values = set(
        feature.unique()
    )


    if not found_values.issubset(
        {
            2019,
            2020
        }
    ):

        raise ValueError(
            f"{FEATURE_COLUMN} contains unexpected values: "
            f"{found_values}"
        )


    # ========================================================
    # 13. ABSOLUTE COUNT OF EACH YEAR
    # ========================================================

    year_count = (
        feature
        .value_counts()
        .reindex(
            [
                2019,
                2020
            ],
            fill_value=0
        )
    )


    count_2019 = int(
        year_count.loc[2019]
    )


    count_2020 = int(
        year_count.loc[2020]
    )


    total = int(
        year_count.sum()
    )


    if total == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    # ========================================================
    # 14. PERCENTAGE PROPORTION OF EACH YEAR
    # ========================================================

    percentage_2019 = (
        count_2019
        / total
        * 100
    )


    percentage_2020 = (
        count_2020
        / total
        * 100
    )


    # ========================================================
    # 15. IDENTIFY THE MAJORITY AND MINORITY YEARS
    # ========================================================

    if count_2019 >= count_2020:

        majority_year = 2019
        majority_count = count_2019

        minority_year = 2020
        minority_count = count_2020

    else:

        majority_year = 2020
        majority_count = count_2020

        minority_year = 2019
        minority_count = count_2019


    # ========================================================
    # 16. IMBALANCE RATIO
    # ========================================================

    if minority_count > 0:

        imbalance_ratio = (
            majority_count
            / minority_count
        )

    else:

        imbalance_ratio = np.inf


    # ========================================================
    # 17. SHANNON ENTROPY
    # ========================================================

    proportions = (
        year_count
        / total
    )


    shannon_entropy = -sum(
        proportion
        * np.log2(
            proportion
        )

        for proportion in proportions

        if proportion > 0
    )


    # ========================================================
    # 18. FORMAT TEXTUAL RESULTS
    # ========================================================

    if np.isfinite(
        imbalance_ratio
    ):

        imbalance_ratio_text = (
            f"{imbalance_ratio:.2f}:1"
        )


        imbalance_interpretation = (
            f"There is approximately "
            f"1 transaction recorded in {minority_year} "
            f"for every {imbalance_ratio:.2f} transactions "
            f"recorded in {majority_year}."
        )


        ir_text = (
            f"{imbalance_ratio:.4f}"
        )


    else:

        imbalance_ratio_text = (
            "Undefined"
        )


        imbalance_interpretation = (
            "The imbalance ratio could not be calculated "
            "because one of the years contains no observations."
        )


        ir_text = (
            "Undefined"
        )


    # ========================================================
    # 19. DATA USED IN THE DISTRIBUTION CHART
    # ========================================================

    labels = [
        "2019",
        "2020"
    ]


    values = [
        count_2019,
        count_2020
    ]


    # ========================================================
    # 20. CREATE THE DISTRIBUTION CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not distribution_chart_exists:

        fig, ax = plt.subplots(
            figsize=(
                8,
                6
            )
        )


        bars = ax.bar(
            labels,
            values
        )


        ax.set_title(
            "Transaction distribution by year"
        )


        ax.set_xlabel(
            "Year"
        )


        ax.set_ylabel(
            "Number of transactions"
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        # ----------------------------------------------------
        # Y AXIS WITHOUT THOUSANDS SEPARATORS
        #
        # Example:
        # 1000000
        #
        # Not:
        # 1,000,000
        # ----------------------------------------------------

        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(
                        y
                    )
                )
            )
        )


        # ----------------------------------------------------
        # ABSOLUTE COUNT ABOVE EACH BAR
        # ----------------------------------------------------

        for bar, value in zip(
            bars,
            values
        ):

            ax.text(
                bar.get_x()
                + bar.get_width() / 2,

                value,

                str(
                    value
                ),

                ha="center",
                va="bottom"
            )


        fig.tight_layout()


        fig.savefig(
            DISTRIBUTION_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nDistribution chart created:"
        )

        print(
            DISTRIBUTION_CHART_PATH
        )


    else:

        print(
            "\nDistribution chart already exists."
        )

        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 21. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            image_base64 = (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


        return image_base64


    # ========================================================
    # 22. CREATE THE HTML REPORT
    #
    # Only if the HTML does not already exist.
    # ========================================================

    if not html_exists:

        distribution_chart_base64 = (
            image_to_base64(
                DISTRIBUTION_CHART_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1100px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 25px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 10px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 35px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.summary {{
    margin-bottom: 40px;
}}

</style>

</head>


<body>


<!-- ========================================================
     TITLE
========================================================= -->


<h1>
Individual Exploratory Analysis — {FEATURE_COLUMN}
</h1>


<p>

The variable <strong>{FEATURE_COLUMN}</strong>
represents the year in which each transaction
was recorded in the dataset.

</p>


<p>

The values observed in this feature are:

</p>


<ul>

<li>
<strong>2019</strong>
</li>

<li>
<strong>2020</strong>
</li>

</ul>


<!-- ========================================================
     1. ABSOLUTE COUNT
========================================================= -->


<h2>
1. Absolute number of transactions by year
</h2>


<table>

<thead>

<tr>

<th>
Year
</th>

<th>
Count
</th>

</tr>

</thead>


<tbody>


<tr>

<td>
2019
</td>

<td>
{count_2019}
</td>

</tr>


<tr>

<td>
2020
</td>

<td>
{count_2020}
</td>

</tr>


<tr>

<td>
<strong>Total</strong>
</td>

<td>
<strong>{total}</strong>
</td>

</tr>


</tbody>

</table>


<!-- ========================================================
     2. PERCENTAGE PROPORTION
========================================================= -->


<h2>
2. Percentage proportion of transactions by year
</h2>


<table>

<thead>

<tr>

<th>
Year
</th>

<th>
Count
</th>

<th>
Percentage
</th>

</tr>

</thead>


<tbody>


<tr>

<td>
2019
</td>

<td>
{count_2019}
</td>

<td>
{percentage_2019:.6f}%
</td>

</tr>


<tr>

<td>
2020
</td>

<td>
{count_2020}
</td>

<td>
{percentage_2020:.6f}%
</td>

</tr>


</tbody>

</table>


<!-- ========================================================
     3. IMBALANCE RATIO
========================================================= -->


<h2>
3. Imbalance ratio
</h2>


<p class="result">

{imbalance_ratio_text}

</p>


<p>

The year with the largest number
of observations was
<strong>{majority_year}</strong>.

</p>


<p>

<strong>
{imbalance_interpretation}
</strong>

</p>


<!-- ========================================================
     4. IMBALANCE RATIO (IR)
========================================================= -->


<h2>
4. Imbalance Ratio (IR)
</h2>


<p class="result">

IR = {ir_text}

</p>


<p>

The Imbalance Ratio was calculated
as the ratio between the number
of observations in the most frequent year
and the number of observations
in the least frequent year.

</p>


<p>

A value close to <strong>1</strong>
indicates similar frequencies between
the two years.

Progressively larger values indicate
a greater difference between
the number of observations.

</p>


<!-- ========================================================
     5. SHANNON ENTROPY
========================================================= -->


<h2>
5. Shannon Entropy
</h2>


<p class="result">

Entropy = {shannon_entropy:.6f} bits

</p>


<p>

Shannon Entropy provides a measure
of how balanced the distribution
of observations is between the two years.

</p>


<ul>


<li>

Values close to
<strong>0</strong>
indicate a greater concentration
of observations in only one year.

</li>


<li>

Values close to
<strong>1</strong>
indicate a more balanced distribution
between 2019 and 2020.

</li>


</ul>


<!-- ========================================================
     6. ABSOLUTE DISTRIBUTION
========================================================= -->


<h2>
6. Transaction distribution by year
</h2>


<p>

The chart presents the absolute number
of transactions recorded in 2019 and 2020.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{distribution_chart_base64}"
    alt="Transaction distribution between 2019 and 2020"
>

</div>


<!-- ========================================================
     7. SUMMARY
========================================================= -->


<h2>
7. Summary of results
</h2>


<div class="summary">

<ul>


<li>

<strong>
Total observations:
</strong>

{total}

</li>


<li>

<strong>
Transactions in 2019:
</strong>

{count_2019}

</li>


<li>

<strong>
Transactions in 2020:
</strong>

{count_2020}

</li>


<li>

<strong>
Percentage in 2019:
</strong>

{percentage_2019:.6f}%

</li>


<li>

<strong>
Percentage in 2020:
</strong>

{percentage_2020:.6f}%

</li>


<li>

<strong>
Most frequent year:
</strong>

{majority_year}

</li>


<li>

<strong>
Least frequent year:
</strong>

{minority_year}

</li>


<li>

<strong>
Imbalance ratio:
</strong>

{imbalance_ratio_text}

</li>


<li>

<strong>
Imbalance Ratio:
</strong>

{ir_text}

</li>


<li>

<strong>
Shannon Entropy:
</strong>

{shannon_entropy:.6f} bits

</li>


</ul>

</div>


</body>

</html>
"""


        # ====================================================
        # 23. SAVE THE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )

        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )

        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 24. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature

    gc.collect()


    # ========================================================
    # 25. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "ANALYSIS COMPLETED"
    )

    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )

    print(
        HTML_PATH
    )


    print(
        "\nPNG:"
    )

    print(
        DISTRIBUTION_CHART_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Distribution chart: Will be created

Distribution chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/trans_year_be/trans_year_be_distribution.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/trans_year_be/analysis_trans_year_be.html

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/trans_year_be

HTML:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/trans_year_be/analysis_trans_year_be.html

PNG:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/trans_year_be/trans_year_be_distribution.png


### <span style="color:white"> SEND_GENDER_BE </span> ###

In [8]:
# ============================================================
# 01. ANALYSIS SETTINGS
#
# FOR THE NEXT FEATURES, CHANGE MAINLY THESE TWO LINES
# ============================================================

ENCODING_TYPE = "binary_encoding"
FEATURE_NAME = "send_gender_be"


# Actual column name inside the dataset
FEATURE_COLUMN = FEATURE_NAME.upper()


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / ENCODING_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


DISTRIBUTION_CHART_NAME = (
    f"{FEATURE_NAME}_distribution"
)


DISTRIBUTION_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{DISTRIBUTION_CHART_NAME}.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


distribution_chart_exists = (
    DISTRIBUTION_CHART_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and distribution_chart_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )

    print(
        "No analysis or file creation is required."
    )

    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )

    print(
        "\nExisting files:"
    )

    print(
        HTML_PATH
    )

    print(
        DISTRIBUTION_CHART_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )

    print(
        "=" * 100
    )

    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )

    print(
        "Distribution chart:",
        "Already exists"
        if distribution_chart_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. VALIDATE THE FEATURE
    # ========================================================

    if feature.isna().any():

        raise ValueError(
            f"{FEATURE_COLUMN} contains missing values."
        )


    found_values = set(
        feature.unique()
    )


    if not found_values.issubset(
        {
            "F",
            "M"
        }
    ):

        raise ValueError(
            f"{FEATURE_COLUMN} contains unexpected values: "
            f"{found_values}"
        )


    # ========================================================
    # 13. ABSOLUTE COUNT OF EACH CATEGORY
    # ========================================================

    category_count = (
        feature
        .value_counts()
        .reindex(
            [
                "F",
                "M"
            ],
            fill_value=0
        )
    )


    female_count = int(
        category_count.loc["F"]
    )


    male_count = int(
        category_count.loc["M"]
    )


    total = int(
        category_count.sum()
    )


    if total == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    # ========================================================
    # 14. PERCENTAGE PROPORTION OF EACH CATEGORY
    # ========================================================

    female_percentage = (
        female_count
        / total
        * 100
    )


    male_percentage = (
        male_count
        / total
        * 100
    )


    # ========================================================
    # 15. IDENTIFY THE MAJORITY AND MINORITY CATEGORIES
    # ========================================================

    if female_count >= male_count:

        majority_category = "F"

        majority_gender = (
            "Female"
        )

        majority_count = (
            female_count
        )


        minority_category = "M"

        minority_gender = (
            "Male"
        )

        minority_count = (
            male_count
        )


    else:

        majority_category = "M"

        majority_gender = (
            "Male"
        )

        majority_count = (
            male_count
        )


        minority_category = "F"

        minority_gender = (
            "Female"
        )

        minority_count = (
            female_count
        )


    # ========================================================
    # 16. IMBALANCE RATIO
    # ========================================================

    if minority_count > 0:

        imbalance_ratio = (
            majority_count
            / minority_count
        )

    else:

        imbalance_ratio = np.inf


    # ========================================================
    # 17. SHANNON ENTROPY
    # ========================================================

    proportions = (
        category_count
        / total
    )


    shannon_entropy = -sum(
        proportion
        * np.log2(
            proportion
        )

        for proportion in proportions

        if proportion > 0
    )


    # ========================================================
    # 18. FORMAT TEXTUAL RESULTS
    # ========================================================

    if np.isfinite(
        imbalance_ratio
    ):

        imbalance_ratio_text = (
            f"{imbalance_ratio:.2f}:1"
        )


        imbalance_interpretation = (
            f"There is approximately "
            f"1 {minority_gender.lower()} observation "
            f"for every {imbalance_ratio:.2f} "
            f"{majority_gender.lower()} observations."
        )


        ir_text = (
            f"{imbalance_ratio:.4f}"
        )


    else:

        imbalance_ratio_text = (
            "Undefined"
        )


        imbalance_interpretation = (
            "The imbalance ratio could not be calculated "
            "because one of the categories contains "
            "no observations."
        )


        ir_text = (
            "Undefined"
        )


    # ========================================================
    # 19. DATA USED IN THE DISTRIBUTION CHART
    # ========================================================

    labels = [
        "Female (F)",
        "Male (M)"
    ]


    values = [
        female_count,
        male_count
    ]


    # ========================================================
    # 20. CREATE THE DISTRIBUTION CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not distribution_chart_exists:

        fig, ax = plt.subplots(
            figsize=(
                8,
                6
            )
        )


        bars = ax.bar(
            labels,
            values
        )


        ax.set_title(
            "Distribution of observations by gender"
        )


        ax.set_xlabel(
            "Gender"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        # ----------------------------------------------------
        # Y AXIS WITHOUT THOUSANDS SEPARATORS
        #
        # Example:
        # 1000000
        #
        # Not:
        # 1,000,000
        # ----------------------------------------------------

        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(
                        y
                    )
                )
            )
        )


        # ----------------------------------------------------
        # ABSOLUTE COUNT ABOVE EACH BAR
        # ----------------------------------------------------

        for bar, value in zip(
            bars,
            values
        ):

            ax.text(
                bar.get_x()
                + bar.get_width() / 2,

                value,

                str(
                    value
                ),

                ha="center",
                va="bottom"
            )


        fig.tight_layout()


        fig.savefig(
            DISTRIBUTION_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nDistribution chart created:"
        )

        print(
            DISTRIBUTION_CHART_PATH
        )


    else:

        print(
            "\nDistribution chart already exists."
        )

        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 21. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            image_base64 = (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


        return image_base64


    # ========================================================
    # 22. CREATE THE HTML REPORT
    #
    # Only if the HTML does not already exist.
    # ========================================================

    if not html_exists:

        distribution_chart_base64 = (
            image_to_base64(
                DISTRIBUTION_CHART_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1100px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 25px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 10px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 35px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.summary {{
    margin-bottom: 40px;
}}

</style>

</head>


<body>


<!-- ========================================================
     TITLE
========================================================= -->


<h1>
Individual Exploratory Analysis — {FEATURE_COLUMN}
</h1>


<p>

The variable <strong>{FEATURE_COLUMN}</strong>
represents the registered gender of the sender
associated with each transaction.

</p>


<p>

The observed values in this feature are:

</p>


<ul>

<li>
<strong>F:</strong> Female
</li>

<li>
<strong>M:</strong> Male
</li>

</ul>


<!-- ========================================================
     1. ABSOLUTE COUNT
========================================================= -->


<h2>
1. Absolute count by gender
</h2>


<table>

<thead>

<tr>

<th>
Category
</th>

<th>
Gender
</th>

<th>
Count
</th>

</tr>

</thead>


<tbody>


<tr>

<td>
F
</td>

<td>
Female
</td>

<td>
{female_count}
</td>

</tr>


<tr>

<td>
M
</td>

<td>
Male
</td>

<td>
{male_count}
</td>

</tr>


<tr>

<td>
-
</td>

<td>
<strong>Total</strong>
</td>

<td>
<strong>{total}</strong>
</td>

</tr>


</tbody>

</table>


<!-- ========================================================
     2. PERCENTAGE PROPORTION
========================================================= -->


<h2>
2. Percentage proportion by gender
</h2>


<table>

<thead>

<tr>

<th>
Category
</th>

<th>
Gender
</th>

<th>
Count
</th>

<th>
Percentage
</th>

</tr>

</thead>


<tbody>


<tr>

<td>
F
</td>

<td>
Female
</td>

<td>
{female_count}
</td>

<td>
{female_percentage:.6f}%
</td>

</tr>


<tr>

<td>
M
</td>

<td>
Male
</td>

<td>
{male_count}
</td>

<td>
{male_percentage:.6f}%
</td>

</tr>


</tbody>

</table>


<!-- ========================================================
     3. IMBALANCE RATIO
========================================================= -->


<h2>
3. Imbalance ratio
</h2>


<p class="result">

{imbalance_ratio_text}

</p>


<p>

The category with the largest number
of observations was

<strong>
{majority_gender} ({majority_category})
</strong>.

</p>


<p>

<strong>
{imbalance_interpretation}
</strong>

</p>


<!-- ========================================================
     4. IMBALANCE RATIO (IR)
========================================================= -->


<h2>
4. Imbalance Ratio (IR)
</h2>


<p class="result">

IR = {ir_text}

</p>


<p>

The Imbalance Ratio was calculated
as the ratio between the number of observations
in the most frequent category
and the number of observations
in the least frequent category.

</p>


<p>

A value close to <strong>1</strong>
indicates similar frequencies
between the two categories.

Progressively larger values indicate
a greater difference between
the number of observations.

</p>


<!-- ========================================================
     5. SHANNON ENTROPY
========================================================= -->


<h2>
5. Shannon Entropy
</h2>


<p class="result">

Entropy = {shannon_entropy:.6f} bits

</p>


<p>

Shannon Entropy provides a measure
of how balanced the distribution
of observations is between
the Female and Male categories.

</p>


<ul>


<li>

Values close to
<strong>0</strong>
indicate a greater concentration
of observations in a single category.

</li>


<li>

Values close to
<strong>1</strong>
indicate a more balanced distribution
between the two categories.

</li>


</ul>


<!-- ========================================================
     6. ABSOLUTE DISTRIBUTION
========================================================= -->


<h2>
6. Distribution of observations by gender
</h2>


<p>

The chart presents the absolute number
of observations belonging to the
Female and Male categories
in the variable
<strong>{FEATURE_COLUMN}</strong>.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{distribution_chart_base64}"
    alt="Distribution of {FEATURE_COLUMN}"
>

</div>


<!-- ========================================================
     7. SUMMARY
========================================================= -->


<h2>
7. Summary of results
</h2>


<div class="summary">

<ul>


<li>

<strong>
Total observations:
</strong>

{total}

</li>


<li>

<strong>
Female:
</strong>

{female_count}

</li>


<li>

<strong>
Male:
</strong>

{male_count}

</li>


<li>

<strong>
Female percentage:
</strong>

{female_percentage:.6f}%

</li>


<li>

<strong>
Male percentage:
</strong>

{male_percentage:.6f}%

</li>


<li>

<strong>
Most frequent category:
</strong>

{majority_gender}
({majority_category})

</li>


<li>

<strong>
Least frequent category:
</strong>

{minority_gender}
({minority_category})

</li>


<li>

<strong>
Imbalance ratio:
</strong>

{imbalance_ratio_text}

</li>


<li>

<strong>
Imbalance Ratio:
</strong>

{ir_text}

</li>


<li>

<strong>
Shannon Entropy:
</strong>

{shannon_entropy:.6f} bits

</li>


</ul>

</div>


</body>

</html>
"""


        # ====================================================
        # 23. SAVE THE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )

        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )

        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 24. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature

    gc.collect()


    # ========================================================
    # 25. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "ANALYSIS COMPLETED"
    )

    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )

    print(
        HTML_PATH
    )


    print(
        "\nPNG:"
    )

    print(
        DISTRIBUTION_CHART_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Distribution chart: Will be created

Distribution chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/send_gender_be/send_gender_be_distribution.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/send_gender_be/analysis_send_gender_be.html

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/send_gender_be

HTML:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/send_gender_be/analysis_send_gender_be.html

PNG:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/send_gender_be/send_gender_be_distribution.png


## <span style="color:green"> FREQUÊNCY ENCODING WITH FALLBACK (1/N_TOTAL) </span> ##

### <span style="color:black"> TRANS_NUM_CARD_FEWF </span> ###

### <span style="color:white"> RECIVE_LOC_FEWF </span> ###

### <span style="color:black"> SEND_JOB_FEWF </span> ###

### <span style="color:white"> SEND_NAME_FEWF </span> ###

## <span style="color:green"> ONE-HOT ENCODING WITH IGNORE </span> ##

### <span style="color:black"> RECIVE_CATEGORY_OHEWI </span> ###

### <span style="color:white"> TRANS_WEEK_OHEWI </span> ###

## <span style="color:green"> CYCLICAL ENCODING USING SINE AND COSINE </span> ##

### <span style="color:black"> TRANS_MONTH_SEN + TRANS_MONTH_COS </span> ###

### <span style="color:white"> TRANS_HOUR_SEN + TRANS_HOUR_COS  </span> ###

## <span style="color:green"> COMUM FEATURES </span> ##

### <span style="color:black"> SEND_AGE </span> ###

### <span style="color:white"> TRANS_DAY </span> ###

### <span style="color:black"> RECIVE_LONG </span> ###

### <span style="color:white"> RECIVE_LAT </span> ###

### <span style="color:black"> SEND_POP_REGISTER </span> ###

### <span style="color:white"> SEND_LONG_REGISTER </span> ###

### <span style="color:black"> SEND_LAT_REGISTER </span> ###

### <span style="color:white"> TRANS_VALUE </span> ###